# Actuator-free baseline evaluation

검증된 dataset과 별도 no-target 프레임에서 기존 YOLO/CNN을 평가합니다. 이 노트북은 Pi, GPIO, 서보 또는 발사를 제어하지 않습니다.

## 1. 최신 main과 의존성 준비

Colab Secret `GITHUB_TOKEN`에는 이 비공개 저장소 읽기 권한만 부여합니다.

In [ ]:
import io
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import requests
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN is required")
headers = {"Authorization": f"Bearer {token}", "X-GitHub-Api-Version": "2022-11-28"}
commit_response = requests.get(
    "https://api.github.com/repos/kw-physical-ai/rubber-band-gun-ai/commits/main",
    headers=headers, timeout=60,
)
commit_response.raise_for_status()
commit_sha = commit_response.json()["sha"]
archive_response = requests.get(
    f"https://api.github.com/repos/kw-physical-ai/rubber-band-gun-ai/zipball/{commit_sha}",
    headers=headers, timeout=60,
)
archive_response.raise_for_status()
extract_root = Path("/content/rubber-band-gun-ai-download")
repo_root = Path("/content/rubber-band-gun-ai")
shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(repo_root, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(archive_response.content)) as archive:
    archive.extractall(extract_root)
shutil.move(str(next(extract_root.iterdir())), repo_root)
del token, headers, archive_response, commit_response
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(repo_root / "requirements/colab.txt"),
])
print(f"Repository ready: {commit_sha}")

## 2. Drive와 평가 입력 지정

기준값은 결과를 보기 전에 calibration과 허용 오차에 맞춰 Issue에 기록합니다. `locked-test`는 후보와 기준값을 고정하고 다른 팀원의 승인을 받은 뒤에만 사용합니다.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
workspace_root = Path("/content/drive/MyDrive/rubber-band-gun-ai-data")

dataset_id = "dataset-CHANGE-ME"
run_id = "run-baseline-CHANGE-ME"
split = "validation"
yolo_model = workspace_root / "reference-models/yolo_target.pt"
yaw_model = workspace_root / "reference-models/best_yaw_cnn.pth"
no_target_bundle = workspace_root / "pi-upload/evaluation/no-target-CHANGE-ME"
target_annotations = workspace_root / "evaluation/target-annotations-CHANGE-ME.csv"
target_recall_min = 0.0  # CHANGE ME before viewing predictions
no_target_fpr_max = 0.0  # CHANGE ME before viewing predictions
yaw_mae_max = 0.0  # CHANGE ME from calibrated physical tolerance
target_iou_min = 0.5
locked_test_approved_by = ""
locked_test_approval_reference = ""  # Issue/PR URL or stable ID

if "CHANGE-ME" in dataset_id or "CHANGE-ME" in run_id:
    raise ValueError("Choose immutable dataset and run IDs")
if yaw_mae_max <= 0 or target_recall_min <= 0:
    raise ValueError("Freeze non-placeholder acceptance thresholds first")
for required in (yolo_model, yaw_model, no_target_bundle, target_annotations):
    if not required.exists():
        raise FileNotFoundError(required)

## 3. 평가 실행 및 run 재검증

결과는 `runs/<run-id>/`에 한 번만 생성되며 모델·dataset·no-target 이미지와 결과 파일의 SHA-256이 기록됩니다.

In [ ]:
command = [
    sys.executable, str(repo_root / "scripts/evaluate-baseline.py"),
    "--dataset", str(workspace_root / "processed" / dataset_id),
    "--split", split,
    "--no-target-bundle", str(no_target_bundle),
    "--target-annotations", str(target_annotations),
    "--target-iou-min", str(target_iou_min),
    "--yolo-model", str(yolo_model),
    "--yaw-model", str(yaw_model),
    "--output-root", str(workspace_root / "runs"),
    "--run-id", run_id,
    "--source-commit", commit_sha,
    "--target-recall-min", str(target_recall_min),
    "--no-target-fpr-max", str(no_target_fpr_max),
    "--yaw-mae-max", str(yaw_mae_max),
]
if split == "locked-test":
    if not locked_test_approved_by or not locked_test_approval_reference:
        raise ValueError("Locked test requires a second-person approval record")
    command += [
        "--allow-locked-test",
        "--locked-test-approved-by", locked_test_approved_by,
        "--locked-test-approval-reference", locked_test_approval_reference,
    ]
subprocess.check_call(command)
subprocess.check_call([
    sys.executable, str(repo_root / "scripts/check-baseline-run.py"),
    str(workspace_root / "runs" / run_id),
])